## Installing the dqx

In [0]:
%pip install databricks-labs-dqx

## Restarting the kernel

In [0]:
dbutils.library.restartPython()

## Importing dependencies

In [0]:
from databricks.labs.dqx import *
from databricks.labs.dqx.config import *
from databricks.labs.dqx.engine import *
from databricks.labs.dqx.rule import *
from databricks.sdk import *

## Reading data from the raw_posts table

In [0]:
raw_posts_df = spark.read.table("rsadhya_data_platform_tutorial.bronze.raw_posts")
display(raw_posts_df.limit(10))

In [0]:
from databricks.labs.dqx import check_funcs
from databricks.labs.dqx.engine import DQEngine
from databricks.labs.dqx.rule import DQForEachColRule, DQRowRule
from databricks.sdk import WorkspaceClient

dq_engine = DQEngine(WorkspaceClient())

checks = [
    *DQForEachColRule(
        columns=["Id","CreationDate"],
        criticality="error",
        check_func = check_funcs.is_not_null
    ).get_rules(),
    DQRowRule(
        name="creation_date_not_in_future",
        criticality="error",
        check_func=check_funcs.is_not_in_future,
        column="CreationDate"
    ),
    DQRowRule(
        name="post_type_id_allowed_values",
        criticality="warn",
        check_func=check_funcs.is_in_list,
        column="PostTypeId",
        check_func_kwargs={"allowed": ["1","2","3","4"]}
    )
]

valid_df, quarantined_df = dq_engine.apply_checks_and_split(raw_posts_df, checks)

In [0]:
display(quarantined_df.limit(10))